In [0]:
%pip install openpyxl==3.1.5

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Importing required libraries

import re
import os
import warnings
from openpyxl import load_workbook

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    LongType,
    DateType
)

In [0]:
# Define catalog, schema, and raw data paths
catalog = "pei_assignment"

bronze_schema = f"{catalog}.bronze"

raw_volume_path = "/Volumes/pei_assignment/bronze/raw_data"

products_path = f"{raw_volume_path}/Products.csv"
customers_path = f"{raw_volume_path}/Customer.xlsx"
orders_path = f"{raw_volume_path}/Orders.json"

products_table = f"{bronze_schema}.raw_products"
customers_table = f"{bronze_schema}.raw_customers"
orders_table = f"{bronze_schema}.raw_orders"

# Create Bronze and Silver schemas if they do not already exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.silver")

DataFrame[]

In [0]:
# Define source columns for each dataset
products_columns = [
    "Product_ID",
    "Category",
    "Sub_Category",
    "Product_Name",
    "State",
    "Price_per_product"
]

customer_columns = [
    "Customer_ID",
    "Customer_Name",
    "email",
    "phone",
    "address",
    "Segment",
    "Country",
    "City",
    "State",
    "Postal_Code",
    "Region"
]

orders_columns = [
    "Row_ID",
    "Order_ID",
    "Order_Date",
    "Ship_Date",
    "Ship_Mode",
    "Customer_ID",
    "Product_ID",
    "Quantity",
    "Price",
    "Discount",
    "Profit"
]


# Create a schema with all columns defined as nullable strings
def create_string_schema(columns):
    return StructType([
        StructField(column, StringType(), True)
        for column in columns
    ])


# Create individual schemas for each source dataset
products_schema = create_string_schema(products_columns)
customer_schema = create_string_schema(customer_columns)
orders_schema = create_string_schema(orders_columns)

In [0]:
# Common Utility functions
# Standardize DataFrame column names
def col_name_formatter(df):
    formatted_columns = [
        re.sub(r"[^a-zA-Z0-9]+", "_", column.strip()).strip("_")
        for column in df.columns
    ]

    normalized_columns = [
        column.lower() for column in formatted_columns
    ]

    if len(normalized_columns) != len(set(normalized_columns)):
        raise ValueError(
            "Duplicate column names found after standardization."
        )

    return df.toDF(*formatted_columns)


# Validate required columns and handle unexpected columns
def validate_columns(df, expected_columns, source_name):
    actual_columns = df.columns

    missing_columns = [
        column for column in expected_columns
        if column not in actual_columns
    ]

    unexpected_columns = [
        column for column in actual_columns
        if column not in expected_columns
    ]

    if missing_columns:
        raise ValueError(
            f"Required columns missing in {source_name}: "
            f"{missing_columns}"
        )

    if unexpected_columns:
        warnings.warn(
            f"Unexpected columns found in {source_name}: "
            f"{unexpected_columns}. "
            "These columns will be ignored during ingestion.",
            UserWarning,
            stacklevel=2
        )

    return df.select(*expected_columns)


# Check whether the source DataFrame contains records
def validate_non_empty(df, source_name):
    if df.isEmpty():
        print(
            f"No records found in {source_name}. "
            "Skipping ingestion for this dataset."
        )
        return False

    return True


# Validate source file availability before ingestion
def validate_file_exists(file_path, source_name):
    if not os.path.isfile(file_path):
        raise FileNotFoundError(
            f"Source file not found for {source_name}: {file_path}"
        )

    return True

In [0]:
# Unit testing functions
# Test source file existence
def test_file_exists(file_path, source_name):
    assert validate_file_exists(file_path, source_name), (
        f"File existence test failed for {source_name}."
    )

    print(f"PASS: File existence test - {source_name}")


# Test column name standardization and duplicate column validation
def test_column_formatter(df, expected_columns, source_name):
    formatted_df = col_name_formatter(df)

    missing_columns = [
        column for column in expected_columns
        if column not in formatted_df.columns
    ]

    assert not missing_columns, (
        f"Column formatting failed for {source_name}."
        f"Missing columns: {missing_columns}"
    )

    assert len(formatted_df.columns) == len(
        set(column.lower() for column in formatted_df.columns)
    ), (
        f"Duplicate columns found after formatting for {source_name}."
    )

    print(f"PASS: Column formatter test - {source_name}")

    return formatted_df


# Test required column validation and column selection
def test_validate_columns(df, expected_columns, source_name):
    validated_df = validate_columns(
        df,
        expected_columns,
        source_name
    )

    assert validated_df.columns == expected_columns, (
        f"Column validation failed for {source_name}. "
        f"Expected: {expected_columns}. "
        f"Actual: {validated_df.columns}"
    )

    print(f"PASS: Column validation test - {source_name}")

    return validated_df


# Test that the DataFrame contains records
def test_non_empty(df, source_name):
    assert validate_non_empty(df, source_name), (
        f"DataFrame is empty for {source_name}."
    )

    print(f"PASS: Non-empty validation test - {source_name}")


# Test data type casting for order columns
def test_cast_column(df_order):
    df_order = df_order.withColumn("Price", F.col("Price").cast(DoubleType()))
    assert df_order.schema["Price"].dataType.typeName() == "double", "Price column cast failed"

    df_order = df_order.withColumn("Discount", F.col("Discount").cast(DoubleType()))
    assert df_order.schema["Discount"].dataType.typeName() == "double", "Discount column cast failed"

    df_order = df_order.withColumn("Profit", F.col("Profit").cast(DoubleType()))
    assert df_order.schema["Profit"].dataType.typeName() == "double", "Profit column cast failed"

    df_order = df_order.withColumn("Quantity", F.col("Quantity").cast(LongType()))
    assert df_order.schema["Quantity"].dataType.typeName() == "long", "Quantity column cast failed"

    df_order = df_order.withColumn("Row_ID", F.col("Row_ID").cast(LongType()))
    assert df_order.schema["Row_ID"].dataType.typeName() == "long", "Row_ID column cast failed"

    df_order = df_order.withColumn(
        "Order_Date",
        F.to_date(F.to_timestamp("Order_Date", "d/M/yyyy"))
    )
    assert df_order.schema["Order_Date"].dataType.typeName() == "date", "Order_Date cast failed"

    df_order = df_order.withColumn(
        "Ship_Date",
        F.to_date(F.to_timestamp("Ship_Date", "d/M/yyyy"))
    )
    assert df_order.schema["Ship_Date"].dataType.typeName() == "date", "Ship_Date cast failed"

    return df_order


# Test data type casting for product columns
def test_cast_column_to_double(df_product):
    df_product = df_product.withColumn(
        "Price_per_product",
        F.col("Price_per_product").cast(DoubleType())
    )
    assert df_product.schema["Price_per_product"].dataType.typeName() == "double", "Price_per_product column cast failed"

    return df_product

In [0]:
# Data Preprocessing functions
# Normalize customer names
def normalize_customer_name(column):
    return F.when(
        F.trim(F.col(column)) != "",
        F.trim(F.col(column))
    )


# Normalize email addresses and handle invalid values
def normalize_email(column):
    email = F.lower(F.trim(F.col(column)))

    return F.when(
        email.rlike(r"^[^\s@]+@[^\s@]+\.[^\s@]+$"),
        email
    )


# Normalize and validate phone numbers
def normalize_phone_number(column):
    phone = F.trim(F.col(column))

    valid_format = phone.rlike(r"^[+]?[0-9()\s.-]+$")

    normalized_phone = F.regexp_replace(
        phone,
        r"[^0-9]",
        ""
    )

    phone_length = F.length(normalized_phone)

    return F.when(
        valid_format & phone_length.between(7, 15),
        normalized_phone
    )


# Normalize postal codes
def normalize_postal_code(column):
    postal_code = F.upper(
        F.regexp_replace(
            F.col(column),
            r"[^A-Za-z0-9]",
            ""
        )
    )

    return F.when(
        F.length(postal_code) > 0,
        postal_code
    )

### Question 1 - Raw tables for each source dataset

### Product table

In [0]:
# Validate source file availability
test_file_exists(products_path, "Products")

# Read Products CSV file
products_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("sep", ",")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", True)
    .option("mode", "PERMISSIVE")
    .csv(products_path)
)

# Standardize and validate column names
products_df = test_column_formatter(
    products_df, products_schema.fieldNames(), "Products"
)

products_df = test_validate_columns(
    products_df, products_schema.fieldNames(), "Products"
)

# Validate that the DataFrame contains records
test_non_empty(products_df, "Products")

# Cast Price_per_product to DoubleType
products_df = test_cast_column_to_double(products_df)

# Write data into Bronze Delta table
products_df.write.format("delta").mode("overwrite").saveAsTable(products_table)

print("Products Bronze ingestion completed successfully.")

# display(products_df)
products_df.show(20, truncate=False)

PASS: File existence test - Products
PASS: Column formatter test - Products
PASS: Column validation test - Products
PASS: Non-empty validation test - Products
Products Bronze ingestion completed successfully.
+---------------+---------------+------------+-------------------------------------------------------------------+------------+-----------------+
|Product_ID     |Category       |Sub_Category|Product_Name                                                       |State       |Price_per_product|
+---------------+---------------+------------+-------------------------------------------------------------------+------------+-----------------+
|FUR-CH-10002961|Furniture      |Chairs      |Leather Task Chair, Black                                          |New York    |81.882           |
|TEC-AC-10004659|Technology     |Accessories |Imation Secure+ Hardware Encrypted USB 2.0 Flash Drive; 16GB       |Oklahoma    |72.99            |
|OFF-BI-10002824|Office Supplies|Binders     |Recycled Easel 

### Customer Table

In [0]:
# Validate source file availability
test_file_exists(customers_path, "Customers")

# Read customer data from Excel workbook
workbook = load_workbook(
    customers_path,
    read_only=True,
    data_only=True
)

worksheet = workbook["Worksheet"]
customer_data = list(worksheet.values)
workbook.close()

# Standardize Excel column headers
customer_headers = [
    re.sub(r"[^a-zA-Z0-9]+", "_", str(column).strip()).strip("_")
    for column in customer_data[0]
]

# Validate duplicate column names
if len(customer_headers) != len(set(column.lower() for column in customer_headers)):
    raise ValueError("Duplicate columns found in Customers.")

# Identify missing and unexpected columns
missing_columns = [
    column for column in customer_schema.fieldNames()
    if column not in customer_headers
]

unexpected_columns = [
    column for column in customer_headers
    if column not in customer_schema.fieldNames()
]

if missing_columns:
    raise ValueError(f"Missing columns in Customers: {missing_columns}")

if unexpected_columns:
    warnings.warn(
        f"Unexpected columns in Customers: {unexpected_columns}. "
        "These columns will be ignored.",
        UserWarning
    )

# Identify indexes of the expected customer columns
customer_indexes = [
    customer_headers.index(column)
    for column in customer_schema.fieldNames()
]

# Create Spark DataFrame using the predefined customer schema
customers_df = spark.createDataFrame(
    [
        tuple(
            None if row[index] is None else str(row[index])
            for index in customer_indexes
        )
        for row in customer_data[1:]
        if any(value is not None for value in row)
    ],
    schema=customer_schema
)

# Standardize and validate DataFrame columns
customers_df = test_column_formatter(
    customers_df, customer_schema.fieldNames(), "Customers"
)

customers_df = test_validate_columns(
    customers_df, customer_schema.fieldNames(), "Customers"
)

# Validate that the DataFrame contains records
test_non_empty(customers_df, "Customers")

# Write data into Bronze Delta table
customers_df.write.format("delta").mode("overwrite").saveAsTable(customers_table)

print("Customers Bronze ingestion completed successfully.")

# display(customers_df)
customers_df.show(20, truncate=False)

PASS: File existence test - Customers
PASS: Column formatter test - Customers
PASS: Column validation test - Customers
PASS: Non-empty validation test - Customers
Customers Bronze ingestion completed successfully.
+-----------+------------------------+-----------------------------+---------------------+----------------------------------------------------+-----------+-------------+----------------+------------+-----------+-------+
|Customer_ID|Customer_Name           |email                        |phone                |address                                             |Segment    |Country      |City            |State       |Postal_Code|Region |
+-----------+------------------------+-----------------------------+---------------------+----------------------------------------------------+-----------+-------------+----------------+------------+-----------+-------+
|PW-19240   |Pierre Wener            |bettysullivan808@gmail.com   |421.580.0902x9815    |001 Jones Ridges Suite 338\nJohnsonf

### Order Table

In [0]:
# Validate source file availability
test_file_exists(orders_path, "Orders")

# Define source columns
orders_source_columns = [
    "Row ID",
    "Order ID",
    "Order Date",
    "Ship Date",
    "Ship Mode",
    "Customer ID",
    "Product ID",
    "Quantity",
    "Price",
    "Discount",
    "Profit"
]

# Create source schema
orders_source_schema = create_string_schema(orders_source_columns)

# Read Orders JSON file
orders_df = (
    spark.read
    .option("multiLine", True)
    .option("mode", "FAILFAST")
    .schema(orders_source_schema)
    .json(orders_path)
)

# Standardize column names
orders_df = test_column_formatter(
    orders_df,
    orders_schema.fieldNames(),
    "Orders"
)

# Validate required columns
orders_df = test_validate_columns(
    orders_df,
    orders_schema.fieldNames(),
    "Orders"
)

# Validate that the DataFrame contains records
test_non_empty(orders_df, "Orders")

# Cast columns using the existing function
orders_df = test_cast_column(orders_df)

# Write data into Bronze Delta table
orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(orders_table)

print("Orders Bronze ingestion completed successfully.")

# display(orders_df)
orders_df.show(20, truncate=False)


PASS: File existence test - Orders
PASS: Column formatter test - Orders
PASS: Column validation test - Orders
PASS: Non-empty validation test - Orders
Orders Bronze ingestion completed successfully.
+------+--------------+----------+----------+--------------+-----------+---------------+--------+------+--------+------+
|Row_ID|Order_ID      |Order_Date|Ship_Date |Ship_Mode     |Customer_ID|Product_ID     |Quantity|Price |Discount|Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+--------+------+--------+------+
|1     |CA-2016-122581|2016-08-21|2016-08-25|Standard Class|JK-15370   |FUR-CH-10002961|7       |573.17|0.3     |63.69 |
|2     |CA-2017-117485|2017-09-23|2017-09-29|Standard Class|BD-11320   |TEC-AC-10004659|4       |291.96|0.0     |102.19|
|3     |US-2016-157490|2016-10-06|2016-10-07|First Class   |LB-16795   |OFF-BI-10002824|4       |17.0  |0.7     |-14.92|
|4     |CA-2015-111703|2015-07-02|2015-07-09|Standard Class|KB-16315   |OFF

In [0]:
# Unit testing for the Bronze tables
# Unit tests for Products Bronze table
def test_products_bronze_table():
    products_bronze_df = spark.table(products_table)

    expected_types = {
        "Product_ID": "string",
        "Category": "string",
        "Sub_Category": "string",
        "Product_Name": "string",
        "State": "string",
        "Price_per_product": "double"
    }

    # Validate column names
    assert products_bronze_df.columns == products_schema.fieldNames()

    # Validate data types
    for column, expected_type in expected_types.items():
        actual_type = products_bronze_df.schema[column].dataType.typeName()

        assert actual_type == expected_type, (
            f"{column} expected {expected_type}, got {actual_type}"
        )

    # Validate record count
    assert products_bronze_df.count() == products_df.count()
    assert products_bronze_df.count() > 0

    print("PASS: Products Bronze table tests")


# Unit tests for Customers Bronze table
def test_customers_bronze_table():
    customers_bronze_df = spark.table(customers_table)

    expected_types = {
        column: "string"
        for column in customer_schema.fieldNames()
    }

    # Validate column names
    assert customers_bronze_df.columns == customer_schema.fieldNames()

    # Validate data types
    for column, expected_type in expected_types.items():
        actual_type = customers_bronze_df.schema[column].dataType.typeName()

        assert actual_type == expected_type, (
            f"{column} expected {expected_type}, got {actual_type}"
        )

    # Validate record count
    assert customers_bronze_df.count() == customers_df.count()
    assert customers_bronze_df.count() > 0

    print("PASS: Customers Bronze table tests")


# Unit tests for Orders Bronze table
def test_orders_bronze_table():
    orders_bronze_df = spark.table(orders_table)

    # Validate table existence
    assert spark.catalog.tableExists(orders_table)

    # Validate column names
    assert orders_bronze_df.columns == orders_schema.fieldNames()

    # Validate data types
    expected_types = {
        "Row_ID": "long",
        "Order_ID": "string",
        "Order_Date": "date",
        "Ship_Date": "date",
        "Ship_Mode": "string",
        "Customer_ID": "string",
        "Product_ID": "string",
        "Quantity": "long",
        "Price": "double",
        "Discount": "double",
        "Profit": "double"
    }

    for column, expected_type in expected_types.items():
        actual_type = orders_bronze_df.schema[column].dataType.typeName()
        assert actual_type == expected_type, (
            f"{column} expected {expected_type}, got {actual_type}"
        )

    # Validate record count
    assert orders_bronze_df.count() == orders_df.count()

    # Validate table is not empty
    assert orders_bronze_df.count() > 0

    print("PASS: Orders Bronze table tests")


# Execute Bronze table tests
test_products_bronze_table()
test_customers_bronze_table()
test_orders_bronze_table()

PASS: Products Bronze table tests
PASS: Customers Bronze table tests
PASS: Orders Bronze table tests


In [0]:
# Data Quality for the Bronze Tables
# Read the Bronze tables
products_df = spark.table(products_table)
customers_df = spark.table(customers_table)
orders_df = spark.table(orders_table)


# 1. Product Data Quality Checks
print("Product Data Quality Checks")

# Check null Product_ID records
null_product_ids_df = products_df.filter(
    F.col("Product_ID").isNull()
)

print("Records with null Product_ID:")
# display(null_product_ids_df)
null_product_ids_df.show(20, truncate=False)

# Identify duplicate Product_ID values
duplicate_products_df = (
    products_df
    .groupBy("Product_ID")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.col("count").desc())
)

print("Duplicate Product_ID values:")
# display(duplicate_products_df)
duplicate_products_df.show(20, truncate=False)

# Display complete records for duplicate Product_IDs
duplicate_product_details_df = (
    products_df.alias("p")
    .join(
        duplicate_products_df.select("Product_ID").alias("d"),
        F.col("p.Product_ID") == F.col("d.Product_ID"),
        "inner"
    )
    .select("p.*")
)

print("Complete duplicate product records:")
# display(duplicate_product_details_df)
duplicate_product_details_df.show(20, truncate=False)

# Check null values in important product attributes
null_product_attributes_df = products_df.filter(
    F.col("Category").isNull()
    | F.col("Sub_Category").isNull()
    | F.col("Product_Name").isNull()
)

print("Products with null important attributes:")
# display(null_product_attributes_df)
null_product_attributes_df.show(20, truncate=False)


# 2. Customer Data Quality Checks
print("Customer Data Quality Checks")

# Check null Customer_ID records
null_customer_ids_df = customers_df.filter(
    F.col("Customer_ID").isNull()
)

print("Records with null Customer_ID:")
# display(null_customer_ids_df)
null_customer_ids_df.show(20, truncate=False)

# Identify duplicate Customer_ID values
duplicate_customers_df = (
    customers_df
    .groupBy("Customer_ID")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.col("count").desc())
)

print("Duplicate Customer_ID values:")
# display(duplicate_customers_df)
duplicate_customers_df.show(20, truncate=False)

# Display complete records for duplicate Customer_IDs
duplicate_customer_details_df = (
    customers_df.alias("c")
    .join(
        duplicate_customers_df.select("Customer_ID").alias("d"),
        F.col("c.Customer_ID") == F.col("d.Customer_ID"),
        "inner"
    )
    .select("c.*")
)

print("Complete duplicate customer records:")
# display(duplicate_customer_details_df)
duplicate_customer_details_df.show(20, truncate=False)

# Check null values in important customer attributes
null_customer_attributes_df = customers_df.filter(
    F.col("Customer_Name").isNull()
)

print("Customers with null important attributes:")
# display(null_customer_attributes_df)
null_customer_attributes_df.show(20, truncate=False)

# Customer Email Validation
customer_email_df = customers_df.withColumn(
    "Normalized_Email",
    F.when(
        F.lower(F.trim(F.col("email"))).rlike(
            r"^[^\s@]+@[^\s@]+\.[^\s@]+$"
        ),
        F.lower(F.trim(F.col("email")))
    )
)

invalid_email_addresses_df = customer_email_df.filter(
    F.col("email").isNotNull()
    & (F.trim(F.col("email")) != "")
    & F.col("Normalized_Email").isNull()
)

print("Customers with invalid email addresses:")
# display(
#     invalid_email_addresses_df.select(
#         "Customer_ID",
#         "email"
#     )
# )
(
    invalid_email_addresses_df.select(
        "Customer_ID",
        "email"
    )
    .show(20, truncate=False)
)

print("Total invalid email addresses:", invalid_email_addresses_df.count())


# Customer Phone Number Validation
customer_phone_df = customers_df.withColumn(
    "Normalized_Phone",
    F.when(
        F.trim(F.col("phone")).rlike(r"^[+]?[0-9()\s.-]+$"),
        F.regexp_replace(
            F.trim(F.col("phone")),
            r"[^0-9]",
            ""
        )
    )
)

customer_phone_df = customer_phone_df.withColumn(
    "Normalized_Phone",
    F.when(
        F.length(F.col("Normalized_Phone")).between(7, 15),
        F.col("Normalized_Phone")
    )
)

invalid_phone_numbers_df = customer_phone_df.filter(
    F.col("phone").isNotNull()
    & (F.trim(F.col("phone")) != "")
    & F.col("Normalized_Phone").isNull()
)

print("Customers with invalid phone numbers:")
# display(
#     invalid_phone_numbers_df.select(
#         "Customer_ID",
#         "phone"
#     )
# )
(
    invalid_phone_numbers_df.select(
        "Customer_ID",
        "phone"
    )
    .show(20, truncate=False)
)

print("Total invalid phone numbers:", invalid_phone_numbers_df.count())


# Customer Postal Code Validation
customer_postal_df = customers_df.withColumn(
    "Normalized_Postal_Code",
    F.upper(
        F.regexp_replace(
            F.trim(F.col("Postal_Code")),
            r"[^A-Za-z0-9]",
            ""
        )
    )
)

invalid_postal_codes_df = customer_postal_df.filter(
    F.col("Postal_Code").isNotNull()
    & (F.trim(F.col("Postal_Code")) != "")
    & (F.length(F.col("Normalized_Postal_Code")) == 0)
)

print("Customers with invalid postal codes:")
# display(
#     invalid_postal_codes_df.select(
#         "Customer_ID",
#         "Postal_Code"
#     )
# )
(
    invalid_postal_codes_df.select(
        "Customer_ID",
        "Postal_Code"
    )
    .show(20, truncate=False)
)    

print("Total invalid postal codes:", invalid_postal_codes_df.count())


# 3. Order Data Quality Checks
print("Order Data Quality Checks")

# Check null Row_ID records
null_row_ids_df = orders_df.filter(
    F.col("Row_ID").isNull()
)

print("Records with null Row_ID:")
# display(null_row_ids_df)
null_row_ids_df.show(20, truncate=False)

# Identify duplicate Row_ID values
duplicate_orders_df = (
    orders_df
    .groupBy("Row_ID")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.col("count").desc())
)

print("Duplicate Row_ID values:")
# display(duplicate_orders_df)
duplicate_orders_df.show(20, truncate=False)

# Display complete records for duplicate Row_IDs
duplicate_order_details_df = (
    orders_df.alias("o")
    .join(
        duplicate_orders_df.select("Row_ID").alias("d"),
        F.col("o.Row_ID") == F.col("d.Row_ID"),
        "inner"
    )
    .select("o.*")
    .orderBy("Row_ID")
)

print("Complete duplicate order records:")
# display(duplicate_order_details_df)
duplicate_order_details_df.show(20, truncate=False)

# Check null values in important order attributes
null_order_attributes_df = orders_df.filter(
    F.col("Order_ID").isNull()
    | F.col("Customer_ID").isNull()
    | F.col("Product_ID").isNull()
)

print("Orders with null important attributes:")
# display(null_order_attributes_df)
null_order_attributes_df.show(20, truncate=False)

Product Data Quality Checks
Records with null Product_ID:
+----------+--------+------------+------------+-----+-----------------+
|Product_ID|Category|Sub_Category|Product_Name|State|Price_per_product|
+----------+--------+------------+------------+-----+-----------------+
+----------+--------+------------+------------+-----+-----------------+

Duplicate Product_ID values:
+---------------+-----+
|Product_ID     |count|
+---------------+-----+
|TEC-AC-10003832|2    |
|OFF-AP-10000576|2    |
|FUR-CH-10001146|2    |
|OFF-ST-10004950|2    |
|OFF-PA-10000357|2    |
|FUR-FU-10004864|2    |
|OFF-PA-10002377|2    |
|FUR-FU-10001473|2    |
|OFF-PA-10001166|2    |
|TEC-AC-10002049|2    |
|FUR-FU-10004270|2    |
|FUR-FU-10004091|2    |
|OFF-PA-10002195|2    |
|TEC-PH-10001795|2    |
|FUR-CH-10002961|2    |
|OFF-BI-10002026|2    |
|OFF-BI-10004632|2    |
|TEC-MA-10001148|2    |
|OFF-PA-10000659|2    |
|OFF-PA-10001970|2    |
+---------------+-----+
only showing top 20 rows
Complete duplicate prod

### Question 2

In [0]:
# Read the Bronze tables
customers_df = spark.table(customers_table)
orders_df = spark.table(orders_table)
products_df = spark.table(products_table)


# Join Customers, Orders and Products
df_enriched_customer_products = (
    customers_df.alias("c")
    .join(
        orders_df.alias("o"),
        F.col("c.Customer_ID") == F.col("o.Customer_ID"),
        "inner"
    )
    .join(
        products_df.alias("p"),
        F.col("o.Product_ID") == F.col("p.Product_ID"),
        "inner"
    )
    .select(
        F.col("c.Customer_ID"),

        # Customer details
        normalize_customer_name("c.Customer_Name").alias("Customer_Name"),
        normalize_email("c.email").alias("email"),
        normalize_phone_number("c.phone").alias("phone"),
        F.col("c.address"),
        F.col("c.Segment"),
        F.col("c.Country"),
        F.col("c.City"),
        F.col("c.State").alias("Customer_State"),
        normalize_postal_code("c.Postal_Code").alias("Postal_Code"),
        F.col("c.Region"),

        # Product details
        F.col("p.Product_ID"),
        F.col("p.Category"),
        F.col("p.Sub_Category"),
        F.col("p.Product_Name"),
        F.col("p.State").alias("Product_State"),
        F.col("p.Price_per_product")
    )
)

# Validate the enriched DataFrame
test_non_empty(
    df_enriched_customer_products,
    "Enriched Customer and Product"
)

# Display enriched data
# display(df_enriched_customer_products)
df_enriched_customer_products.show(20, truncate=False)

# Define Silver table name
enriched_customer_products_table = (
    f"{catalog}.silver.enriched_customer_products"
)

# Save the enriched DataFrame as a Silver Delta table
(
    df_enriched_customer_products.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(enriched_customer_products_table)
)

print("Enriched Customer and Product table created successfully.")

PASS: Non-empty validation test - Enriched Customer and Product
+-----------+------------------+--------------------------------+----------+----------------------------------------------------------+-----------+-------------+-------------+--------------+-----------+-------+---------------+---------------+------------+-------------------------------------------------------------------+-------------+-----------------+
|Customer_ID|Customer_Name     |email                           |phone     |address                                                   |Segment    |Country      |City         |Customer_State|Postal_Code|Region |Product_ID     |Category       |Sub_Category|Product_Name                                                       |Product_State|Price_per_product|
+-----------+------------------+--------------------------------+----------+----------------------------------------------------------+-----------+-------------+-------------+--------------+-----------+-------+--------------

In [0]:
# Unit testing for enriched_customer_products_table
# Load the saved Silver table
enriched_df = spark.table(enriched_customer_products_table)

# Test 1: Check if the table exists
assert spark.catalog.tableExists(
    enriched_customer_products_table
), "Enriched Customer and Product table does not exist."


# Test 2: Check expected columns
expected_schema = [
    "Customer_ID",
    "Customer_Name",
    "email",
    "phone",
    "address",
    "Segment",
    "Country",
    "City",
    "Customer_State",
    "Postal_Code",
    "Region",
    "Product_ID",
    "Category",
    "Sub_Category",
    "Product_Name",
    "Product_State",
    "Price_per_product"
]

assert enriched_df.columns == expected_schema, (
    f"Schema mismatch. Expected: {expected_schema}, "
    f"Actual: {enriched_df.columns}"
)


# Test 3: Check whether the table contains records
row_count = enriched_df.count()

assert row_count > 0, (
    "Enriched Customer and Product table is empty."
)


# Test 4: Check for duplicate column names
assert len(enriched_df.columns) == len(
    set(column.lower() for column in enriched_df.columns)
), "Duplicate column names found."


# Test 5: Validate Customer_ID and Product_ID
null_customer_ids = enriched_df.filter(
    F.col("Customer_ID").isNull()
).count()

null_product_ids = enriched_df.filter(
    F.col("Product_ID").isNull()
).count()

assert null_customer_ids == 0, (
    "Null Customer_ID values found."
)

assert null_product_ids == 0, (
    "Null Product_ID values found."
)


# Test 6: Validate the join result against source tables
expected_count = (
    customers_df.alias("c")
    .join(
        orders_df.alias("o"),
        F.col("c.Customer_ID") == F.col("o.Customer_ID"),
        "inner"
    )
    .join(
        products_df.alias("p"),
        F.col("o.Product_ID") == F.col("p.Product_ID"),
        "inner"
    )
    .count()
)

assert row_count == expected_count, (
    f"Record count mismatch. Expected: {expected_count}, "
    f"Actual: {row_count}"
)

print("PASS: Enriched Customer and Product table tests")

PASS: Enriched Customer and Product table tests


In [0]:

# Data Quality checks for enriched_customer_products_table:
# Check duplicate Product_ID values in the Products table
duplicate_products_df = (
    products_df
    .groupBy("Product_ID")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.col("count").desc())
)

# Display duplicate Product_ID values
# display(duplicate_products_df)
duplicate_products_df.show(20, truncate=False)

# Display all records for duplicate Product_ID values
duplicate_product_details_df = (
    products_df.alias("p")
    .join(
        duplicate_products_df.select("Product_ID").alias("d"),
        F.col("p.Product_ID") == F.col("d.Product_ID"),
        "inner"
    )
    .select("p.*")
    .orderBy("Product_ID")
)

# display(duplicate_product_details_df)
duplicate_product_details_df.show(20, truncate=False)


+---------------+-----+
|Product_ID     |count|
+---------------+-----+
|TEC-AC-10003832|2    |
|OFF-AP-10000576|2    |
|FUR-CH-10001146|2    |
|OFF-ST-10004950|2    |
|OFF-PA-10000357|2    |
|FUR-FU-10004864|2    |
|OFF-PA-10002377|2    |
|FUR-FU-10001473|2    |
|OFF-PA-10001166|2    |
|TEC-AC-10002049|2    |
|FUR-FU-10004270|2    |
|FUR-FU-10004091|2    |
|OFF-PA-10002195|2    |
|TEC-PH-10001795|2    |
|FUR-CH-10002961|2    |
|OFF-BI-10002026|2    |
|OFF-BI-10004632|2    |
|TEC-MA-10001148|2    |
|OFF-PA-10000659|2    |
|OFF-PA-10001970|2    |
+---------------+-----+
only showing top 20 rows
+---------------+---------------+------------+---------------------------------------------------------------------------------------+------------+-----------------+
|Product_ID     |Category       |Sub_Category|Product_Name                                                                           |State       |Price_per_product|
+---------------+---------------+------------+---------------------

### Question 3

In [0]:
# Read data from Bronze tables
orders_df = spark.table(orders_table)
customers_df = spark.table(customers_table)
products_df = spark.table(products_table)

# Join Orders with Customers and Products
df_enriched_order_customer_product = (
    orders_df.alias("o")
    .join(
        customers_df.alias("c"),
        F.col("o.Customer_ID") == F.col("c.Customer_ID"),
        "inner"
    )
    .join(
        products_df.alias("p"),
        F.col("o.Product_ID") == F.col("p.Product_ID"),
        "inner"
    )
    .select(
        # Order Information
        F.col("o.Row_ID"),
        F.col("o.Order_ID"),
        F.col("o.Order_Date"),
        F.col("o.Ship_Date"),
        F.col("o.Ship_Mode"),
        F.col("o.Customer_ID"),
        F.col("o.Product_ID"),
        F.col("o.Quantity"),
        F.col("o.Price"),
        F.col("o.Discount"),
        F.round(F.col("o.Profit"), 2).alias("Profit"),

        # Customer Information
        normalize_customer_name("c.Customer_Name").alias("Customer_Name"),
        F.col("c.Country"),

        # Product Information
        F.col("p.Category"),
        F.col("p.Sub_Category")
    )
)

# Display the enriched data
# display(df_enriched_order_customer_product)
df_enriched_order_customer_product.show(20, truncate=False)

# Define Silver table name
enriched_order_customer_product_table = f"{catalog}.silver.enriched_order_customer_product"

# Save the enriched DataFrame as a Silver Delta table
(
    df_enriched_order_customer_product
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(enriched_order_customer_product_table)
)

print(
    f"Table created successfully: "
    f"{enriched_order_customer_product_table}"
)

+------+--------------+----------+----------+--------------+-----------+---------------+--------+------+--------+------+------------------+-------------+---------------+------------+
|Row_ID|Order_ID      |Order_Date|Ship_Date |Ship_Mode     |Customer_ID|Product_ID     |Quantity|Price |Discount|Profit|Customer_Name     |Country      |Category       |Sub_Category|
+------+--------------+----------+----------+--------------+-----------+---------------+--------+------+--------+------+------------------+-------------+---------------+------------+
|1     |CA-2016-122581|2016-08-21|2016-08-25|Standard Class|JK-15370   |FUR-CH-10002961|7       |573.17|0.3     |63.69 |Jay Kimmel        |United States|Furniture      |Chairs      |
|2     |CA-2017-117485|2017-09-23|2017-09-29|Standard Class|BD-11320   |TEC-AC-10004659|4       |291.96|0.0     |102.19|Bi1l Donatelli    |United States|Technology     |Accessories |
|3     |US-2016-157490|2016-10-06|2016-10-07|First Class   |LB-16795   |OFF-BI-100028

In [0]:
# Unit Test for enriched_order_customer_product_table
# Load the saved enriched table
enriched_df = spark.table(enriched_order_customer_product_table)


# Test 1: Verify that the enriched table exists
def test_enriched_table_exists():
    assert spark.catalog.tableExists(
        enriched_order_customer_product_table
    ), "Enriched table was not created"

    print("PASS: Enriched table exists")


# Test 2: Verify the expected columns
def test_enriched_table_schema():
    expected_columns = [
        "Row_ID",
        "Order_ID",
        "Order_Date",
        "Ship_Date",
        "Ship_Mode",
        "Customer_ID",
        "Product_ID",
        "Quantity",
        "Price",
        "Discount",
        "Profit",
        "Customer_Name",
        "Country",
        "Category",
        "Sub_Category"
    ]

    assert enriched_df.columns == expected_columns, (
        f"Unexpected schema: {enriched_df.columns}"
    )

    print("PASS: Enriched table schema validation")


# Test 3: Verify that the enriched table contains records
def test_enriched_table_not_empty():
    row_count = enriched_df.count()

    assert row_count > 0, "Enriched table is empty"

    print(f"PASS: Enriched table contains {row_count} records")


# Test 4: Verify Profit is rounded to 2 decimal places
def test_profit_rounding():
    invalid_profit_count = (
        enriched_df
        .filter(
            F.col("Profit").isNotNull() &
            (F.col("Profit") != F.round(F.col("Profit"), 2))
        )
        .count()
    )

    assert invalid_profit_count == 0, (
        f"Found {invalid_profit_count} records "
        "where Profit is not rounded to 2 decimal places"
    )

    print("PASS: Profit rounding validation")


# Test 5: Verify record count against the source joins
def test_enriched_table_record_count():
    expected_count = (
        orders_df.alias("o")
        .join(
            customers_df.alias("c"),
            F.col("o.Customer_ID") == F.col("c.Customer_ID"),
            "inner"
        )
        .join(
            products_df.alias("p"),
            F.col("o.Product_ID") == F.col("p.Product_ID"),
            "inner"
        )
        .count()
    )

    actual_count = enriched_df.count()

    assert actual_count == expected_count, (
        f"Record count mismatch. "
        f"Expected: {expected_count}, Actual: {actual_count}"
    )

    print("PASS: Enriched table record count validation")


# Execute all tests
test_enriched_table_exists()
test_enriched_table_schema()
test_enriched_table_not_empty()
test_profit_rounding()
test_enriched_table_record_count()

PASS: Enriched table exists
PASS: Enriched table schema validation
PASS: Enriched table contains 10133 records
PASS: Profit rounding validation
PASS: Enriched table record count validation


In [0]:
# Data Quality Consideration for enriched_order_customer_product_table
# Product_ID values in the source Products dataset are not unique. Some Product_ID values are associated with multiple product records.
# This can result in duplicate records after joining with Orders.
# The source data quality issue is intentionally exposed through dedicated data quality checks.

# Identify duplicate order lines
duplicate_order_line_ids_df = (
    enriched_df
    .groupBy("Row_ID")
    .count()
    .filter(F.col("count") > 1)
)

# Display duplicate Row_ID values
print("Duplicate order line IDs:")
# display(duplicate_order_line_ids_df)
duplicate_order_line_ids_df.show(20, truncate=False)

# Display complete records for duplicate order lines
duplicate_order_line_details_df = (
    enriched_df.alias("e")
    .join(
        duplicate_order_line_ids_df.select("Row_ID").alias("d"),
        on="Row_ID",
        how="inner"
    )
    .orderBy("Row_ID")
)

print("Complete records for duplicate order lines:")
# display(duplicate_order_line_details_df)
duplicate_order_line_details_df.show(20, truncate=False)

# Display duplicate record count
duplicate_order_line_count = duplicate_order_line_ids_df.count()

print(
    f"Number of Row_ID values with duplicate records: "
    f"{duplicate_order_line_count}"
)


# Identify orders without matching Customer records
orders_without_customer_df = (
    orders_df.alias("o")
    .join(
        customers_df.select("Customer_ID").distinct().alias("c"),
        on="Customer_ID",
        how="left_anti"
    )
)

print("Orders without matching Customer records:")
# display(orders_without_customer_df)
orders_without_customer_df.show(20, truncate=False)

print(
    "Number of orders without matching customers:",
    orders_without_customer_df.count()
)


# Identify orders without matching Product records
orders_without_product_df = (
    orders_df.alias("o")
    .join(
        products_df.select("Product_ID").distinct().alias("p"),
        on="Product_ID",
        how="left_anti"
    )
)

print("Orders without matching Product records:")
# display(orders_without_product_df)
orders_without_product_df.show(20, truncate=False)

print(
    "Number of orders without matching products:",
    orders_without_product_df.count()
)

Duplicate order line IDs:
+------+-----+
|Row_ID|count|
+------+-----+
|1     |2    |
|48    |2    |
|54    |2    |
|98    |2    |
|158   |2    |
|213   |2    |
|220   |2    |
|230   |2    |
|233   |2    |
|289   |2    |
|330   |2    |
|345   |2    |
|381   |2    |
|403   |2    |
|496   |2    |
|553   |2    |
|556   |2    |
|563   |2    |
|578   |2    |
|580   |2    |
+------+-----+
only showing top 20 rows
Complete records for duplicate order lines:
+------+--------------+----------+----------+--------------+-----------+---------------+--------+------+--------+------+-----------------+-------------+---------------+------------+
|Row_ID|Order_ID      |Order_Date|Ship_Date |Ship_Mode     |Customer_ID|Product_ID     |Quantity|Price |Discount|Profit|Customer_Name    |Country      |Category       |Sub_Category|
+------+--------------+----------+----------+--------------+-----------+---------------+--------+------+--------+------+-----------------+-------------+---------------+------------+

### Question 4

In [0]:
# Read data from Bronze tables
orders_df = spark.table(orders_table)
customers_df = spark.table(customers_table)
products_df = spark.table(products_table)

# Join Orders with Products and Customers
df_profit_by_year_pc_psc_customer = (
    orders_df.alias("o")
    .join(
        products_df.alias("p"),
        F.col("o.Product_ID") == F.col("p.Product_ID"),
        "inner"
    )
    .join(
        customers_df.alias("c"),
        F.col("o.Customer_ID") == F.col("c.Customer_ID"),
        "inner"
    )
    .groupBy(
        F.year(F.col("o.Order_Date")).alias("Year"),
        F.col("p.Category"),
        F.col("p.Sub_Category"),
        F.col("c.Customer_ID"),
        F.col("c.Customer_Name")
    )
    .agg(
        F.round(F.sum(F.col("o.Profit")), 2).alias("Aggregate_Profit")
    )
    .select(
        "Aggregate_Profit",
        "Year",
        "Category",
        "Sub_Category",
        "Customer_ID",
        "Customer_Name"
    )
)

# Display aggregated profit data by year
# display(
#     df_profit_by_year_pc_psc_customer
#     .orderBy(F.col("Year").desc())
# )
(
    df_profit_by_year_pc_psc_customer
        .orderBy(F.col("Year").desc())
        .show(20, truncate=False)
)

# Define Silver aggregate table name
profit_by_year_pc_psc_customer_table = (
    f"{catalog}.silver.profit_by_year_pc_psc_customer"
)

# Save the aggregated DataFrame as a Silver Delta table
(
    df_profit_by_year_pc_psc_customer
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(profit_by_year_pc_psc_customer_table)
)

print(
    f"Aggregate table created successfully: "
    f"{profit_by_year_pc_psc_customer_table}"
)

+----------------+----+---------------+------------+-----------+----------------+
|Aggregate_Profit|Year|Category       |Sub_Category|Customer_ID|Customer_Name   |
+----------------+----+---------------+------------+-----------+----------------+
|58.42           |2017|Furniture      |Tables      |LC-16870   |NULL            |
|43.42           |2017|Office Supplies|Storage     |BW-11110   |Bart Watters    |
|2.1             |2017|Office Supplies|Appliances  |CS-12175   |Charles Sheldon |
|-7.39           |2017|Office Supplies|Appliances  |CD-11980   |Carol Darley    |
|2.04            |2017|Office Supplies|Art         |TA-21385   |Tom Ashbrook    |
|-19.28          |2017|Office Supplies|Storage     |DS-13030   |Darrin Sayre    |
|28.8            |2017|Office Supplies|Paper       |DH-13075   |Dave Hallsten   |
|821.61          |2017|Office Supplies|Binders     |AH-10210   |Alan Hwang      |
|-34.31          |2017|Technology     |Phones      |FH-14350   |Fred Harton     |
|39.64          

In [0]:
# Unit testing for profit_by_year_pc_psc_customer_table
# Load the saved aggregate table
profit_aggregate_df = spark.table(
    profit_by_year_pc_psc_customer_table
)


# Test 1: Verify that the aggregate table exists
def test_profit_aggregate_table_exists():
    assert spark.catalog.tableExists(
        profit_by_year_pc_psc_customer_table
    ), "Aggregate table was not created"

    print("PASS: Aggregate table exists")


# Test 2: Verify the expected schema
def test_profit_aggregate_table_schema():
    expected_schema = [
        "Aggregate_Profit",
        "Year",
        "Category",
        "Sub_Category",
        "Customer_ID",
        "Customer_Name"
    ]

    assert profit_aggregate_df.columns == expected_schema, (
        f"Unexpected schema: {profit_aggregate_df.columns}"
    )

    print("PASS: Aggregate table schema validation")


# Test 3: Verify that the aggregate table contains records
def test_profit_aggregate_table_not_empty():
    row_count = profit_aggregate_df.count()

    assert row_count > 0, "Aggregate table is empty"

    print(f"PASS: Aggregate table contains {row_count} records")


# Test 4: Verify aggregate profit against actual source data
def test_profit_aggregation():
    expected_df = (
        orders_df.alias("o")
        .join(
            products_df.alias("p"),
            F.col("o.Product_ID") == F.col("p.Product_ID"),
            "inner"
        )
        .join(
            customers_df.alias("c"),
            F.col("o.Customer_ID") == F.col("c.Customer_ID"),
            "inner"
        )
        .groupBy(
            F.year(F.col("o.Order_Date")).alias("Year"),
            F.col("p.Category"),
            F.col("p.Sub_Category"),
            F.col("c.Customer_ID"),
            F.col("c.Customer_Name")
        )
        .agg(
            F.round(F.sum(F.col("o.Profit")), 2).alias("Aggregate_Profit")
        )
    )

    actual_df = profit_aggregate_df

    comparison_columns = [
        "Year",
        "Category",
        "Sub_Category",
        "Customer_ID",
        "Customer_Name",
        "Aggregate_Profit"
    ]

    expected_df = expected_df.select(*comparison_columns)
    actual_df = actual_df.select(*comparison_columns)

    assert expected_df.exceptAll(actual_df).count() == 0, (
        "Expected aggregate records are missing from the output table"
    )

    assert actual_df.exceptAll(expected_df).count() == 0, (
        "Unexpected aggregate records found in the output table"
    )

    print("PASS: Profit aggregation validation")


# Test 5: Verify that each grouping combination is unique
def test_profit_aggregate_grouping():
    duplicate_count = (
        profit_aggregate_df
        .groupBy(
            "Year",
            "Category",
            "Sub_Category",
            "Customer_ID",
            "Customer_Name"
        )
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    assert duplicate_count == 0, (
        f"Found {duplicate_count} duplicate grouping combinations"
    )

    print("PASS: Aggregate grouping uniqueness validation")


# Execute all tests
test_profit_aggregate_table_exists()
test_profit_aggregate_table_schema()
test_profit_aggregate_table_not_empty()
test_profit_aggregation()
test_profit_aggregate_grouping()


# display(
#     spark.table(profit_by_year_pc_psc_customer_table)
# )

PASS: Aggregate table exists
PASS: Aggregate table schema validation
PASS: Aggregate table contains 7937 records
PASS: Profit aggregation validation
PASS: Aggregate grouping uniqueness validation


### Question 5

In [0]:
# Profit grouped by year
df_profit_by_year = spark.sql(f"""
    SELECT
        SUM(Aggregate_Profit) AS Profit_By_Year,
        Year
    FROM {profit_by_year_pc_psc_customer_table}
    GROUP BY Year
    ORDER BY Year DESC
""")

print("1. Profit by Year")
# display(df_profit_by_year)
df_profit_by_year.show(20, truncate=False)


# Profit grouped by year and product category
df_profit_by_year_cat = spark.sql(f"""
    SELECT
        SUM(Aggregate_Profit) AS Profit_By_Year_And_Prd_Cat,
        Year,
        Category AS Product_Category
    FROM {profit_by_year_pc_psc_customer_table}
    GROUP BY
        Year,
        Category
    ORDER BY Year DESC
""")

print("2. Profit by Year and Product Category")
# display(df_profit_by_year_cat)
df_profit_by_year_cat.show(20, truncate=False)


# Profit grouped by customer
df_profit_by_customer = spark.sql(f"""
    SELECT
        SUM(Aggregate_Profit) AS Profit_By_Customer,
        Customer_ID,
        Customer_Name
    FROM {profit_by_year_pc_psc_customer_table}
    GROUP BY
        Customer_ID,
        Customer_Name
    ORDER BY Customer_ID
""")

print("3. Profit by Customer")
# display(df_profit_by_customer)
df_profit_by_customer.show(20, truncate=False)


# Profit grouped by year and customer
df_profit_by_year_customer = spark.sql(f"""
    SELECT
        SUM(Aggregate_Profit) AS Profit_By_Year_And_Customer,
        Year,
        Customer_ID,
        Customer_Name
    FROM {profit_by_year_pc_psc_customer_table}
    GROUP BY
        Year,
        Customer_ID,
        Customer_Name
    ORDER BY Year DESC
""")

print("4. Profit by Year and Customer")
# display(df_profit_by_year_customer)
df_profit_by_year_customer.show(20, truncate=False)

1. Profit by Year
+------------------+----+
|Profit_By_Year    |Year|
+------------------+----+
|127175.23000000001|2017|
|68161.13000000014 |2016|
|65706.24999999997 |2015|
|40975.400000000016|2014|
+------------------+----+

2. Profit by Year and Product Category
+--------------------------+----+----------------+
|Profit_By_Year_And_Prd_Cat|Year|Product_Category|
+--------------------------+----+----------------+
|3361.7599999999975        |2017|Furniture       |
|78483.07999999999         |2017|Technology      |
|45330.390000000065        |2017|Office Supplies |
|7750.150000000001         |2016|Furniture       |
|35973.599999999984        |2016|Office Supplies |
|24437.379999999972        |2016|Technology      |
|36823.78999999999         |2015|Technology      |
|25490.340000000037        |2015|Office Supplies |
|3392.1200000000026        |2015|Furniture       |
|22663.759999999987        |2014|Office Supplies |
|-5174.65                  |2014|Furniture       |
|23486.289999999994 

In [0]:
# Check profit by year
def test_profit_by_year():
    expected_schema = ["Profit_By_Year", "Year"]
    actual_schema = df_profit_by_year.columns

    assert actual_schema == expected_schema, (
        f"Expected schema: {expected_schema}, but got: {actual_schema}"
    )

    assert df_profit_by_year.count() > 0, "Profit by year result is empty."

    expected_count = (
        spark.table(profit_by_year_pc_psc_customer_table)
        .select("Year")
        .distinct()
        .count()
    )

    actual_count = df_profit_by_year.select("Year").distinct().count()

    assert actual_count == expected_count, (
        "Year-wise aggregation is missing one or more years."
    )

    print("test_profit_by_year: Passed")


# Check profit by year and product category
def test_profit_by_year_cat():
    expected_schema = [
        "Profit_By_Year_And_Prd_Cat",
        "Year",
        "Product_Category"
    ]
    actual_schema = df_profit_by_year_cat.columns

    assert actual_schema == expected_schema, (
        f"Expected schema: {expected_schema}, but got: {actual_schema}"
    )

    assert df_profit_by_year_cat.count() > 0, (
        "Profit by year and product category result is empty."
    )

    expected_count = (
        spark.table(profit_by_year_pc_psc_customer_table)
        .select("Year", "Category")
        .distinct()
        .count()
    )

    actual_count = (
        df_profit_by_year_cat
        .select("Year", "Product_Category")
        .distinct()
        .count()
    )

    assert actual_count == expected_count, (
        "Year and product category aggregation is missing one or more groups."
    )

    print("test_profit_by_year_cat: Passed")


# Check profit by customer
def test_profit_by_customer():
    expected_schema = [
        "Profit_By_Customer",
        "Customer_ID",
        "Customer_Name"
    ]
    actual_schema = df_profit_by_customer.columns

    assert actual_schema == expected_schema, (
        f"Expected schema: {expected_schema}, but got: {actual_schema}"
    )

    assert df_profit_by_customer.count() > 0, (
        "Profit by customer result is empty."
    )

    expected_count = (
        spark.table(profit_by_year_pc_psc_customer_table)
        .select("Customer_ID", "Customer_Name")
        .distinct()
        .count()
    )

    actual_count = (
        df_profit_by_customer
        .select("Customer_ID", "Customer_Name")
        .distinct()
        .count()
    )

    assert actual_count == expected_count, (
        "Customer aggregation is missing one or more customers."
    )

    print("test_profit_by_customer: Passed")


# Check profit by year and customer
def test_profit_by_year_customer():
    expected_schema = [
        "Profit_By_Year_And_Customer",
        "Year",
        "Customer_ID",
        "Customer_Name"
    ]
    actual_schema = df_profit_by_year_customer.columns

    assert actual_schema == expected_schema, (
        f"Expected schema: {expected_schema}, but got: {actual_schema}"
    )

    assert df_profit_by_year_customer.count() > 0, (
        "Profit by year and customer result is empty."
    )

    expected_count = (
        spark.table(profit_by_year_pc_psc_customer_table)
        .select("Year", "Customer_ID", "Customer_Name")
        .distinct()
        .count()
    )

    actual_count = (
        df_profit_by_year_customer
        .select("Year", "Customer_ID", "Customer_Name")
        .distinct()
        .count()
    )

    assert actual_count == expected_count, (
        "Year and customer aggregation is missing one or more groups."
    )

    print("test_profit_by_year_customer: Passed")


# Run the validation tests for all four aggregations
test_profit_by_year()
test_profit_by_year_cat()
test_profit_by_customer()
test_profit_by_year_customer()

test_profit_by_year: Passed
test_profit_by_year_cat: Passed
test_profit_by_customer: Passed
test_profit_by_year_customer: Passed
